# AgentMeter — measure-vram on Colab L4 (per-model weight footprint)

**Purpose:** capture each of the 5 models' **weight footprint** to complete the
Phase 8 SAW *total device VRAM* criterion. The full-run DB stored only *marginal*
working memory; it never recorded the resident model weights.

This does **NOT** re-run the ~9-hour full benchmark. It loads each of the 5 models
**once** and reads device VRAM before/after — **no scenarios, no generation**
(~10 min total). The existing full-run DB is left untouched; this only writes
`results/model_vram.json`.

> **Must run on an L4** — the SAME GPU + SAME 4bit-nf4 quant as the full run — so
> the weight footprint is consistent with the working-memory figures already in the
> DB. A different GPU (e.g. a T4) makes the capture invalid; Cell 7 flags that.

## 1. GPU check
This **must** be an **L4** (same hardware as the full run). A T4 or other GPU makes
the weight footprint inconsistent with the DB's working-memory numbers — re-run on
an L4 if this isn't one. Runtime → Change runtime type → **L4 GPU**.

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime -> Change runtime type -> L4 GPU (required)'

## 2. Mount Google Drive
For consistency with the full-run notebook. The repo is cloned to
`/content/AgentMeter`; `results/model_vram.json` is written there and downloaded in
Cell 8 (no symlink needed — this run produces a single small JSON).

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/agentmeter_results'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive results dir:', DRIVE_DIR, '(optional backup target)')

## 3. Get the AgentMeter code
Clones and **hard-resets** to `origin/BRANCH` so a stale clone can't linger. HEAD
must be **f79bbe0** (the measure-vram commit) **or newer**. Private repo → paste a
token (hidden); public → press Enter.

In [ ]:
import os, getpass, subprocess
REPO = 'https://github.com/ismazahin/AgentMeter'
BRANCH = 'claude/cool-ride-mitmzl'
gh = getpass.getpass('GitHub token (press Enter if repo is public): ').strip()
url = REPO.replace('https://', f'https://{gh}@') if gh else REPO
if not os.path.isdir('/content/AgentMeter'):
    subprocess.run(['git', 'clone', url + '.git', '/content/AgentMeter'], check=True)
os.chdir('/content/AgentMeter')
subprocess.run(['git', 'remote', 'set-url', 'origin', url + '.git'], check=True)
subprocess.run(['git', 'fetch', '--force', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
print('cwd   :', os.getcwd())
print('branch:', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],capture_output=True,text=True).stdout.strip())
print('HEAD  :', subprocess.run(['git','log','-1','--oneline'],capture_output=True,text=True).stdout.strip())
print('  (HEAD must contain the measure-vram commit f79bbe0 or newer)')

## 4. Install dependencies
Colab already ships a CUDA `torch`; we don't reinstall it. Add the CPU deps plus the
GPU/HF extras and `bitsandbytes` for 4-bit.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q transformers accelerate huggingface_hub sentencepiece pynvml bitsandbytes

## 5. Hugging Face token
Entered via `getpass` or a Colab Secret named `HF_TOKEN` — **not** hardcoded, not
printed. **Accept the gated licences first** on each model's HF page:
`mistralai/Mistral-7B-Instruct-v0.3`, `meta-llama/Meta-Llama-3-8B-Instruct`, and
`google/gemma-2-9b-it` (Qwen and Phi-3 are ungated). Otherwise the load 401s.

In [ ]:
import os, getpass
tok = ''
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not tok:
    tok = getpass.getpass('Enter your HF_TOKEN (hidden): ').strip()
assert tok, 'HF_TOKEN is required for the gated models.'
os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set (', len(tok), 'chars ). Not displayed.')

## 6. Capture weight footprints (load-and-read only)
Loads each of the 5 models once in its own subprocess (fresh CUDA context; each
model's cache freed after), reads device VRAM before/after, and writes
`results/model_vram.json`. **No scenarios, no generation.** ~1–2 min per model.

In [ ]:
!python main.py --config configs/run_full_l4.yaml measure-vram

## 7. Check the capture is valid (hardware = L4)
Prints `results/model_vram.json` and asserts every model was measured on an **L4**.

> **If `consistent_hardware` is false or the label isn't an L4** (e.g. a T4 slipped
> in), the capture is **invalid** — switch Runtime to an L4 and re-run Cells 1–6
> before using this file.

In [ ]:
import json
mv = json.load(open('results/model_vram.json'))
print(json.dumps(mv, indent=2))
print()
hw = mv.get('hardware_labels', [])
print('hardware_labels    :', hw)
print('consistent_hardware:', mv.get('consistent_hardware'))
is_l4 = mv.get('consistent_hardware') and all('L4' in h for h in hw)
if not is_l4:
    print('!' * 72)
    print('INVALID CAPTURE: models were NOT all measured on an L4.')
    print('Switch Runtime -> Change runtime type -> L4, then re-run Cells 1-6.')
    print('!' * 72)
assert is_l4, f'measure-vram must run on an L4 (got {hw}); re-run on an L4 GPU.'
print('OK: all 5 models measured on an L4 — footprints are consistent with the full run.')

## 8. Download the result
Download `results/model_vram.json` (and optionally back it up to Drive). Send this
file back to complete the total-VRAM SAW criterion — no further GPU work is needed.

**When done: Runtime → Disconnect and delete runtime to stop L4 billing.**

In [ ]:
import os, shutil
# optional Drive backup if mounted
drive_dir = '/content/drive/MyDrive/agentmeter_results'
if os.path.isdir(drive_dir):
    try:
        shutil.copy('results/model_vram.json', os.path.join(drive_dir, 'model_vram.json'))
        print('Backed up to', drive_dir)
    except Exception as e:
        print('Drive copy skipped:', e)

from google.colab import files
try:
    files.download('results/model_vram.json')
except Exception as e:
    print('Download manually from the Files panel:', e)
print('\nDone. Runtime -> Disconnect and delete runtime to stop billing.')